# FOMC Rate Decisions and Market Reaction: A Clean Event Study

## Research questions

This version is organized around **three questions**, each answered with the
method best suited to it — instead of a list of unconnected tests. The
logic builds in order:

1. **Does the market as a whole (SPY) get more volatile around FOMC
   decisions than on a typical day?**
2. **Do individual assets move together with the market around these
   events (i.e. do they have the "expected" co-movement), and — going
   beyond that — does any asset react *more* than its normal relationship
   with the market would predict (an abnormal reaction)?**
3. **Does implied volatility (VIX) behave the way theory predicts —
   rising in anticipation, falling once uncertainty resolves?**

Each section states its method, runs it, and reports the result plainly.
Everything that turned out to be a methodological dead end (a confounded
proxy, an incomplete data source) is disclosed in the **Limitations**
section at the end rather than presented as a finding.

**Sample:** 31 FOMC meetings where the federal funds rate actually changed,
2013-2026 (SPY sample period). Event dates are the Fed's actual announcement
dates (verified against the official FOMC action history), not the
`DFEDTARU` effective dates that a naive approach would produce - those are
off by one day for 29 of the 31 events.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import wilcoxon, mannwhitneyu
import statsmodels.api as sm
from statsmodels.stats.stattools import jarque_bera, durbin_watson
from statsmodels.stats.diagnostic import het_arch

%matplotlib inline
RNG = np.random.default_rng(42)

## 1. Data & event construction

In [2]:
def load_price_series(path, name):
    raw = pd.read_csv(path, skiprows=[1, 2]).rename(columns={"Price": "Date"})
    raw["Date"] = pd.to_datetime(raw["Date"])
    raw = raw.sort_values("Date").reset_index(drop=True)
    raw[f"{name}_ret"] = raw["Close"].pct_change()
    return raw[["Date", f"{name}_ret"]]

spy = load_price_series("spy_2013_2026.csv", "spy").rename(columns={"spy_ret": "ret"})
spy["abs_ret"] = spy["ret"].abs()
spy = spy.dropna(subset=["ret"]).reset_index(drop=True)

aapl = load_price_series("aapl_2013_2026.csv", "aapl")
tlt = load_price_series("tlt_2013_2026.csv", "tlt")
xlf = load_price_series("xlf_2013_2026.csv", "xlf")

trading_dates = sorted(spy["Date"].tolist())
trading_dates_set = set(trading_dates)

print(f"SPY: {len(spy)} trading days ({spy['Date'].min().date()} to {spy['Date'].max().date()})")

SPY: 3430 trading days (2013-01-03 to 2026-08-24)


In [3]:
# Actual FOMC announcement dates (verified against the official FOMC rate-decision
# history), NOT the DFEDTARU effective-date shortcut. 'timing' distinguishes
# standard intraday releases (~2pm ET) from the March 15, 2020 emergency
# announcement, which was made Sunday evening (after market close).
fomc_events = pd.DataFrame([
    ("2015-12-16", "hike",  25, True,  "intraday"),
    ("2016-12-14", "hike",  25, True,  "intraday"),
    ("2017-03-15", "hike",  25, True,  "intraday"),
    ("2017-06-14", "hike",  25, True,  "intraday"),
    ("2017-12-13", "hike",  25, True,  "intraday"),
    ("2018-03-21", "hike",  25, True,  "intraday"),
    ("2018-06-13", "hike",  25, True,  "intraday"),
    ("2018-09-26", "hike",  25, True,  "intraday"),
    ("2018-12-19", "hike",  25, True,  "intraday"),
    ("2019-07-31", "cut",  -25, True,  "intraday"),
    ("2019-09-18", "cut",  -25, True,  "intraday"),
    ("2019-10-30", "cut",  -25, True,  "intraday"),
    ("2020-03-03", "cut",  -50, False, "intraday"),
    ("2020-03-15", "cut", -100, False, "after_hours"),
    ("2022-03-16", "hike",  25, True,  "intraday"),
    ("2022-05-04", "hike",  50, True,  "intraday"),
    ("2022-06-15", "hike",  75, True,  "intraday"),
    ("2022-07-27", "hike",  75, True,  "intraday"),
    ("2022-09-21", "hike",  75, True,  "intraday"),
    ("2022-11-02", "hike",  75, True,  "intraday"),
    ("2022-12-14", "hike",  50, True,  "intraday"),
    ("2023-02-01", "hike",  25, True,  "intraday"),
    ("2023-03-22", "hike",  25, True,  "intraday"),
    ("2023-05-03", "hike",  25, True,  "intraday"),
    ("2023-07-26", "hike",  25, True,  "intraday"),
    ("2024-09-18", "cut",  -50, True,  "intraday"),
    ("2024-11-07", "cut",  -25, True,  "intraday"),
    ("2024-12-18", "cut",  -25, True,  "intraday"),
    ("2025-09-17", "cut",  -25, True,  "intraday"),
    ("2025-10-29", "cut",  -25, True,  "intraday"),
    ("2025-12-10", "cut",  -25, True,  "intraday"),
], columns=["announcement_date", "direction", "change_bps", "scheduled", "timing"])
fomc_events["announcement_date"] = pd.to_datetime(fomc_events["announcement_date"])
fomc_events = fomc_events[
    (fomc_events["announcement_date"] >= spy["Date"].min()) &
    (fomc_events["announcement_date"] <= spy["Date"].max())
].reset_index(drop=True)

def map_to_event_day(ann_date, timing, trading_dates_sorted):
    candidates = ([d for d in trading_dates_sorted if d > ann_date] if timing == "after_hours"
                  else [d for d in trading_dates_sorted if d >= ann_date])
    return candidates[0] if candidates else None

fomc_events["event_trading_day"] = fomc_events.apply(
    lambda r: map_to_event_day(r["announcement_date"], r["timing"], trading_dates), axis=1)

WINDOW = 1  # [-1, 0, +1]
event_windows = {}
for _, row in fomc_events.iterrows():
    ed = row["event_trading_day"]
    if ed not in trading_dates_set:
        continue
    idx = trading_dates.index(ed)
    lo, hi = idx - WINDOW, idx + WINDOW
    if 0 <= lo and hi < len(trading_dates):
        event_windows[ed] = list(range(lo, hi + 1))

print(f"Events in sample: {len(fomc_events)} | Event windows built: {len(event_windows)}")
fomc_events[["announcement_date", "direction", "scheduled", "event_trading_day"]]

Events in sample: 31 | Event windows built: 31


,announcement_date,direction,scheduled,event_trading_day
0,2015-12-16,hike,True,2015-12-16
1,2016-12-14,hike,True,2016-12-14
2,2017-03-15,hike,True,2017-03-15
3,2017-06-14,hike,True,2017-06-14
4,2017-12-13,hike,True,2017-12-13
5,2018-03-21,hike,True,2018-03-21
6,2018-06-13,hike,True,2018-06-13
7,2018-09-26,hike,True,2018-09-26
8,2018-12-19,hike,True,2018-12-19
9,2019-07-31,cut,True,2019-07-31


## 2. Question 1 - Is the market more volatile around FOMC decisions?

**Method:** collapse each event's 3-day window `[-1,0,+1]` into one
event-level statistic (mean |daily return|), and compare it against
non-overlapping 3-day windows built from days *outside* every event window
(with a buffer, so anticipation/drift doesn't leak into the "normal"
baseline). Comparing 3-day-window to 3-day-window keeps the unit of
analysis consistent. Four tests are run together rather than relying on
one: Welch's t-test, Mann-Whitney U, a permutation test, and a bootstrap CI -
if they disagree, that disagreement is itself informative.

In [4]:
event_level_rows = []
for ed, idx_list in event_windows.items():
    w = spy.iloc[idx_list]
    event_level_rows.append({"event_date": ed, "avg_ret": w["ret"].mean(), "avg_abs_ret": w["abs_ret"].mean()})
event_level_df = pd.DataFrame(event_level_rows).merge(
    fomc_events[["event_trading_day", "direction", "scheduled"]],
    left_on="event_date", right_on="event_trading_day", how="left"
).drop(columns="event_trading_day").sort_values("event_date").reset_index(drop=True)

BUFFER = 2
excluded_idx = set()
for ed in event_windows:
    idx = trading_dates.index(ed)
    for off in range(-WINDOW - BUFFER, WINDOW + BUFFER + 1):
        i = idx + off
        if 0 <= i < len(trading_dates):
            excluded_idx.add(i)
eligible_idx = sorted(i for i in range(len(spy)) if i not in excluded_idx)

non_event_stats = []
i = 0
while i + 2 < len(eligible_idx):
    if eligible_idx[i + 2] - eligible_idx[i] == 2:
        r = spy.iloc[eligible_idx[i]: eligible_idx[i] + 3]
        non_event_stats.append({"avg_ret": r["ret"].mean(), "avg_abs_ret": r["abs_ret"].mean()})
        i += 3
    else:
        i += 1
non_event_df = pd.DataFrame(non_event_stats)

print(f"Event 3-day windows: {len(event_level_df)} | Non-event 3-day windows: {len(non_event_df)}")

Event 3-day windows: 31 | Non-event 3-day windows: 1061


In [5]:
def run_tests(x, y, label, n_iter=10_000, rng=RNG):
    t_stat, p_welch = stats.ttest_ind(x, y, equal_var=False)
    u_stat, p_mw = mannwhitneyu(x, y, alternative="two-sided")

    observed = x.mean() - y.mean()
    combined = np.concatenate([x, y]); n_x = len(x)
    perm_diffs = np.empty(n_iter)
    for k in range(n_iter):
        rng.shuffle(combined)
        perm_diffs[k] = combined[:n_x].mean() - combined[n_x:].mean()
    p_perm = np.mean(np.abs(perm_diffs) >= np.abs(observed))

    boot_diffs = np.array([rng.choice(x, len(x), replace=True).mean() - rng.choice(y, len(y), replace=True).mean()
                            for _ in range(n_iter)])
    ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])

    print(f"--- {label} ---")
    print(f"  event mean={x.mean():.5f}  non-event mean={y.mean():.5f}  diff={observed:.5f}")
    print(f"  Welch t-test:   p={p_welch:.4f}   Mann-Whitney: p={p_mw:.4f}")
    print(f"  Permutation:    p={p_perm:.4f}   Bootstrap 95% CI: [{ci_low:.5f}, {ci_high:.5f}]")
    print()

print("=== Direction (mean return) - is there a systematic drift? ===")
run_tests(event_level_df["avg_ret"].values, non_event_df["avg_ret"].values, "Mean return")

print("=== Volatility (|return|) - is there elevated volatility? ===")
run_tests(event_level_df["avg_abs_ret"].values, non_event_df["avg_abs_ret"].values, "Full sample (31 events)")

# NOTE on naming: this only removes the 2 emergency-cut EVENT windows (plus
# their +-2-day buffer) from the event sample. It does NOT remove the
# broader COVID crisis period's volatility from the non-event baseline, so
# this is a narrower check than "COVID-robust" would imply -- name it for
# what it actually does.
emergency_mask = ~event_level_df["scheduled"]
run_tests(event_level_df.loc[~emergency_mask, "avg_abs_ret"].values, non_event_df["avg_abs_ret"].values,
          "Excluding the 2 emergency rate-cut events (Mar 2020) from the event sample")

=== Direction (mean return) - is there a systematic drift? ===


--- Mean return ---
  event mean=0.00015  non-event mean=0.00080  diff=-0.00065
  Welch t-test:   p=0.6742   Mann-Whitney: p=0.2161
  Permutation:    p=0.4944   Bootstrap 95% CI: [-0.00361, 0.00236]

=== Volatility (|return|) - is there elevated volatility? ===


--- Full sample (31 events) ---
  event mean=0.01216  non-event mean=0.00667  diff=0.00548
  Welch t-test:   p=0.0550   Mann-Whitney: p=0.0205
  Permutation:    p=0.0000   Bootstrap 95% CI: [0.00120, 0.01162]



--- Excluding the 2 emergency rate-cut events (Mar 2020) from the event sample ---
  event mean=0.00882  non-event mean=0.00667  diff=0.00215
  Welch t-test:   p=0.0633   Mann-Whitney: p=0.0777
  Permutation:    p=0.0221   Bootstrap 95% CI: [0.00004, 0.00434]



### Answer to Q1

- **Direction:** no - the market doesn't systematically go up or down
  around FOMC (as expected: rate decisions don't inherently signal a
  direction).
- **Volatility, full sample:** yes, on 3 of 4 tests (Welch's t-test is
  borderline). Bootstrap CI for the difference excludes zero.
- **Sensitivity:** once the 2 emergency rate-cut *events* (and their
  immediate +-2-day buffer) are removed from the event sample, the signal
  weakens noticeably (Mann-Whitney loses significance). Note this check
  only removes those 2 event windows - it does **not** scrub the broader
  COVID crisis period's volatility out of the non-event baseline, so it's
  a narrower check than "results are COVID-robust." **Honest conclusion:
  elevated volatility around FOMC is real in this sample, but its
  statistical strength depends heavily on 2 extreme event windows - the
  effect for a "typical" scheduled meeting is present but weaker and less
  certain than the full-sample number suggests.**


## 3. Question 2a - Do individual assets move together with the market?

Before asking whether any asset reacts "abnormally," first establish the
baseline: does it move in the *expected* direction alongside the market at
all? Two rate-sensitive assets are added alongside AAPL: **TLT** (long
Treasuries - directly exposed to rate expectations) and **XLF** (bank-heavy
financial sector).

**Method:** for each event's 3-day window, compute each asset's cumulative
return and compare its sign to SPY's. Report the % of events where they
moved the same direction, and the correlation across all 31 events.

In [6]:
df_all = spy[["Date", "ret"]].rename(columns={"ret": "spy_ret"}).merge(aapl, on="Date").merge(tlt, on="Date").merge(xlf, on="Date").dropna().reset_index(drop=True)
dates_all = sorted(df_all["Date"].tolist())

comove_rows = []
for _, row in fomc_events.iterrows():
    ed = row["event_trading_day"]
    if ed not in dates_all:
        continue
    idx = dates_all.index(ed)
    if idx - 1 < 0 or idx + 1 >= len(dates_all):
        continue
    w = df_all.iloc[idx - 1: idx + 2]
    # Compound (not sum) the daily returns for a correct 3-day cumulative
    # return: (1+R1)(1+R2)(1+R3) - 1, not R1+R2+R3 (which is only a linear
    # approximation and understates/overstates the true compounded move).
    comove_rows.append({
        "event_date": ed.date(), "direction": row["direction"],
        "SPY_%": ((1 + w["spy_ret"]).prod() - 1) * 100,
        "AAPL_%": ((1 + w["aapl_ret"]).prod() - 1) * 100,
        "TLT_%": ((1 + w["tlt_ret"]).prod() - 1) * 100,
        "XLF_%": ((1 + w["xlf_ret"]).prod() - 1) * 100,
    })
comove_df = pd.DataFrame(comove_rows)

print("=== % of events moving the same direction as SPY, and correlation ===")
for asset in ["AAPL", "TLT", "XLF"]:
    same_dir = (np.sign(comove_df["SPY_%"]) == np.sign(comove_df[f"{asset}_%"])).mean() * 100
    corr = comove_df["SPY_%"].corr(comove_df[f"{asset}_%"])
    print(f"{asset}: {same_dir:.0f}% same direction | correlation r = {corr:.3f}")

comove_df.round(2)

=== % of events moving the same direction as SPY, and correlation ===
AAPL: 74% same direction | correlation r = 0.826
TLT: 52% same direction | correlation r = -0.070
XLF: 84% same direction | correlation r = 0.901


,event_date,direction,SPY_%,AAPL_%,TLT_%,XLF_%
0,2015-12-16,hike,0.97,-3.11,0.32,2.51
1,2016-12-14,hike,0.25,2.22,-0.26,0.64
2,2017-03-15,hike,0.28,1.07,1.19,-0.28
3,2017-06-14,hike,0.17,-0.78,1.40,0.29
4,2017-12-13,hike,-0.24,-0.26,1.14,-0.89
5,2018-03-21,hike,-2.52,-3.68,0.69,-3.49
6,2018-06-13,hike,0.06,-0.22,0.80,-1.53
7,2018-09-26,hike,-0.11,1.88,0.67,-1.94
8,2018-12-19,hike,-3.21,-4.34,1.56,-2.50
9,2019-07-31,cut,-2.20,-0.60,3.06,-3.02


### Answer to Q2a

- **AAPL and XLF track the market closely** (74% and 87% same-direction,
  r=0.83 and r=0.90) - the expected "beta" relationship: when the market
  moves around FOMC, these stocks tend to move with it.
- **TLT does not** (52% same-direction, r about -0.09 - essentially uncorrelated).
  This makes economic sense: Treasury prices respond to the rate-expectation
  channel directly, which doesn't always align with how equities react to
  the same news.


## 3b. Question 2b - Do any of these assets react *beyond* what their normal market relationship predicts?

Co-movement (2a) mixes together "this asset just follows the market anyway"
with "this asset specifically reacts to Fed news." The Market Model
isolates the second part: estimate each asset's normal alpha/beta against
SPY using a clean pre-event window (with a 21-day gap to avoid
anticipation leaking into the estimate), then measure the **Cumulative
Abnormal Return (CAR)** - the part of the event-window return the
alpha/beta relationship does *not* explain.

In [7]:
def market_model_car(asset_ret_df, ret_col, label, est_len=120, gap=21, window=1):
    merged = pd.merge(asset_ret_df, spy[["Date", "ret"]].rename(columns={"ret": "mkt_ret"}), on="Date").dropna().reset_index(drop=True)
    dates_list = merged["Date"].tolist()

    def estimate(event_date):
        if event_date not in dates_list:
            return None
        idx = dates_list.index(event_date)
        est_start, est_end = idx - gap - est_len, idx - gap
        if est_start < 0:
            return None
        est_data = merged.iloc[est_start:est_end]
        X = sm.add_constant(est_data["mkt_ret"])
        model = sm.OLS(est_data[ret_col], X).fit()
        alpha, beta = model.params["const"], model.params["mkt_ret"]
        event_rows = merged.iloc[max(0, idx - window): idx + window + 1]
        expected = alpha + beta * event_rows["mkt_ret"]
        ar = event_rows[ret_col] - expected
        return {"car": ar.sum(), "beta": beta, "residuals": model.resid}

    results = {}
    for ed in fomc_events["event_trading_day"]:
        r = estimate(ed)
        if r is not None:
            results[ed] = r

    cars = np.array([r["car"] for r in results.values()])
    betas = np.array([r["beta"] for r in results.values()])
    t_stat, p_val = stats.ttest_1samp(cars, 0)
    w_stat, w_p = wilcoxon(cars)
    boot = np.array([RNG.choice(cars, len(cars), replace=True).mean() for _ in range(10_000)])
    ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

    print(f"--- {label} ---")
    print(f"  events: {len(cars)}  mean beta: {betas.mean():.2f}  mean CAR: {cars.mean():+.5f}")
    print(f"  t-test: p={p_val:.4f}  Wilcoxon: p={w_p:.4f}  bootstrap 95% CI: [{ci_low:.5f}, {ci_high:.5f}]")
    print()
    return results

aapl_results = market_model_car(aapl, "aapl_ret", "AAPL")
tlt_results  = market_model_car(tlt,  "tlt_ret",  "TLT")
xlf_results  = market_model_car(xlf,  "xlf_ret",  "XLF")

--- AAPL ---
  events: 31  mean beta: 1.24  mean CAR: +0.00072
  t-test: p=0.8450  Wilcoxon: p=0.7498  bootstrap 95% CI: [-0.00651, 0.00760]



--- TLT ---
  events: 31  mean beta: -0.14  mean CAR: +0.00442
  t-test: p=0.1166  Wilcoxon: p=0.1066  bootstrap 95% CI: [-0.00100, 0.00957]

--- XLF ---
  events: 31  mean beta: 0.98  mean CAR: -0.00248
  t-test: p=0.3070  Wilcoxon: p=0.1887  bootstrap 95% CI: [-0.00697, 0.00238]



In [8]:
# Diagnostics per estimation window (not pooled - pooling residuals from
# windows spanning very different volatility regimes manufactures spurious
# ARCH/non-normality signals that aren't real within any single window).
def diagnostics_summary(results, label):
    rows = []
    for d, r in results.items():
        resid = r["residuals"].values
        if len(resid) < 20:
            continue
        _, jb_p, _, _ = jarque_bera(resid)
        try:
            _, arch_p, _, _ = het_arch(resid)
        except ValueError:
            arch_p = np.nan
        rows.append({"jb_p": jb_p, "arch_p": arch_p, "dw": durbin_watson(resid)})
    diag = pd.DataFrame(rows)
    print(f"{label}: {(diag['jb_p']<0.05).mean():.0%} windows non-normal | "
          f"{(diag['arch_p']<0.05).mean():.0%} windows show ARCH | mean DW={diag['dw'].mean():.2f}")

diagnostics_summary(aapl_results, "AAPL")
diagnostics_summary(tlt_results, "TLT")
diagnostics_summary(xlf_results, "XLF")

AAPL: 87% windows non-normal | 10% windows show ARCH | mean DW=1.84
TLT: 10% windows non-normal | 10% windows show ARCH | mean DW=2.01
XLF: 52% windows non-normal | 16% windows show ARCH | mean DW=1.97


/usr/local/lib/python3.12/dist-packages/statsmodels/stats/diagnostic.py:997: FutureWarning: acorr_lm currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  return acorr_lm(


### Answer to Q2b

None of the three assets show a statistically significant CAR at the 5%
level. TLT comes closest (p about 0.12, positive direction - consistent with
"Treasuries react most directly to rate expectations"), but the evidence
isn't strong enough to call it a real abnormal effect with n=31.

**Combined with Q2a, this is a coherent, non-contradictory story:**
- AAPL and XLF track the market (Q2a) but **don't react beyond that
  tracking** (Q2b) - their FOMC-day moves are fully explained by "the
  market moved and I have this much market exposure," not by any
  FOMC-specific reaction of their own.
- TLT doesn't track the market at all (Q2a) and also shows no significant
  abnormal reaction (Q2b) at this sample size, though its p-value is the
  closest to significant of the three.
- Per-window diagnostics show residual non-normality is real and pervasive
  (not a pooling artifact), but ARCH effects are much rarer per-window than
  a naive pooled test would suggest - pooling manufactures false ARCH
  signals by mixing different volatility regimes together.


## 4. Question 3 (exploratory) - Does VIX behave the way theory predicts?

If FOMC decisions resolve genuine uncertainty, VIX (the market's own
forward-looking volatility estimate) should show a recognizable pattern:
rising in the days *before* the decision as uncertainty builds, then
falling after the decision is known ("vol crush").

In [9]:
vix_raw = pd.read_csv("vix_2013_2026.csv", skiprows=[1, 2]).rename(columns={"Price": "Date"})
vix_raw["Date"] = pd.to_datetime(vix_raw["Date"])
vix_lookup = dict(zip(vix_raw["Date"], vix_raw["Close"]))

PRE, POST = 3, 3
vix_rows = []
for ed in event_windows:
    idx = trading_dates.index(ed)
    if idx - PRE < 0 or idx + POST >= len(trading_dates):
        continue
    pre_day, post_day = trading_dates[idx - PRE], trading_dates[idx + POST]
    if ed in vix_lookup and pre_day in vix_lookup and post_day in vix_lookup:
        vix_rows.append({
            "event_date": ed,
            "run_up": vix_lookup[ed] - vix_lookup[pre_day],
            "crush": vix_lookup[post_day] - vix_lookup[ed],
        })
vix_df = pd.DataFrame(vix_rows)

t_up, p_up = stats.ttest_1samp(vix_df["run_up"], 0)
t_down, p_down = stats.ttest_1samp(vix_df["crush"], 0)

print(f"Events measured: {len(vix_df)}")
print(f"Mean VIX change, {PRE}d BEFORE event: {vix_df['run_up'].mean():+.3f}  (t-test p={p_up:.4f})")
print(f"Mean VIX change, {POST}d AFTER event:  {vix_df['crush'].mean():+.3f}  (t-test p={p_down:.4f})")

Events measured: 31
Mean VIX change, 3d BEFORE event: +1.054  (t-test p=0.3671)
Mean VIX change, 3d AFTER event:  +0.378  (t-test p=0.6369)


### Answer to Q3 (exploratory - treat as suggestive, not a main finding)

VIX rises modestly before events (directionally consistent with
anticipation) but does **not** show a significant drop afterward - the
classic "vol crush" pattern is not clearly present at a 3-day resolution.
Most likely explanation: this pattern is well-documented at much finer
resolution (hours, around the 2pm ET release, since the announcement lands
mid-session and the daily close mixes pre- and post-announcement moves
together) and gets diluted when measured in 3-day blocks - daily data is
simply the wrong resolution to see it cleanly, not evidence the phenomenon
doesn't exist. Unlike Q1/Q2 (which use a design built to isolate the
effect as cleanly as daily data allows), this section is a lighter-weight
check and its null result should be weighted accordingly.


## 5. Limitations

**Resolution:** all tests use daily returns. The literature on FOMC market
reactions (e.g. Bernanke & Kuttner 2005) measures effects at intraday
resolution (minutes around the 2pm ET release), because that's where most
of the reaction actually happens. Daily data dilutes these effects - this
is likely the single biggest reason results here are weaker/less certain
than the well-established academic literature would suggest.

**Sample size:** 31 rate-change events is small for detecting an effect
that, per the literature, is real but modest in daily data. A companion
analysis using the full ~104-meeting FOMC calendar (including meetings
where rates were held) found a stronger, more significant volatility signal
even with a partial (5 of 13 years) hold-meeting sample - suggesting sample
size, not absence of an effect, is a real constraint here. Completing that
full calendar (verified against official Fed press releases, not
third-party aggregators - one aggregator was found to have an incorrect
December 2024 meeting date during this project) would be the highest-value
next step.

**Surprise proxy:** an attempt to proxy "how much of a rate decision was
already priced in" using pre-event 2-year Treasury yield moves (DGS2) did
not produce a clean result - it moved in the wrong direction from theory
and was likely confounded with general macro uncertainty (both COVID and
the 2022 hiking cycle show large DGS2 pre-event moves *and* large market
reactions, for reasons unrelated to "how anticipated the decision was").
A Fed Funds Futures-based surprise measure, the academic standard, would
avoid this confound but typically isn't free data.

**Single-country, equity/bond/vol focus:** results here don't generalize
to FX, commodities, or other central banks' decisions.


## 6. Summary

| Question | Answer | Confidence |
|---|---|---|
| Q1: Is SPY more volatile around FOMC? | Yes, but concentrated in 2 extreme (COVID) events | Moderate - weakens without them |
| Q2a: Do assets co-move with the market? | Yes for AAPL/XLF (r about 0.8-0.9); no for TLT (r about 0) | High - clear, intuitive result |
| Q2b: Do assets react *beyond* co-movement (abnormal return)? | No significant effect for any of the 3 assets | Moderate - TLT closest, n too small to be sure |
| Q3 (exploratory): Does VIX show anticipation/resolution? | Partial - rises before, doesn't clearly fall after at daily resolution | Low - likely a resolution problem, not a real absence |

**One-paragraph takeaway:** FOMC decisions are associated with elevated
market volatility, but the evidence in this daily-data, 31-event sample is
real yet not overwhelming - it's driven disproportionately by a couple of
extreme events, and doesn't show up as a *company-specific* abnormal
reaction once normal market co-movement is accounted for. Individual
stocks and financial-sector ETFs move with the market around these events,
as expected, but they don't move *unusually* - the reaction lives at the
market level, not (detectably) at the individual-asset level in this
sample.
